[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/apis/04-google-gemini.ipynb)

# Google Gemini API — Notebook interactivo

Texto, visión, contexto largo y function calling con la API de Google Gemini.

In [ ]:
# pip install google-generativeai python-dotenv
import os
import google.generativeai as genai

genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))

MODELO_FLASH = 'gemini-1.5-flash'
MODELO_PRO = 'gemini-1.5-pro'

modelo = genai.GenerativeModel(MODELO_FLASH)
print(f'Gemini {MODELO_FLASH} listo')

## 1. Generación de texto básica

In [ ]:
from google.generativeai.types import GenerationConfig

config = GenerationConfig(
    temperature=0.3,
    max_output_tokens=512,
    top_p=0.9,
)

respuesta = modelo.generate_content(
    '¿Cuáles son las principales diferencias entre Gemini Flash y Gemini Pro?',
    generation_config=config,
)

print(respuesta.text)
print(f'\nTokens: entrada={respuesta.usage_metadata.prompt_token_count}, '
      f'salida={respuesta.usage_metadata.candidates_token_count}')

## 2. Chat con historial

In [ ]:
chat = modelo.start_chat(history=[])

def enviar_mensaje(mensaje: str) -> str:
    resp = chat.send_message(mensaje)
    return resp.text

# Conversación de ejemplo
print('Turn 1:')
print(enviar_mensaje('Soy desarrollador Python y quiero aprender sobre IA generativa. ¿Por dónde empiezo?'))

print('\nTurn 2:')
print(enviar_mensaje('¿Qué diferencia hay entre usar la API directamente vs usar un framework como LangChain?'))

print(f'\nHistorial: {len(chat.history)} mensajes')

## 3. Análisis de imágenes (visión)

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def analizar_imagen_url(url: str, pregunta: str) -> str:
    resp_http = requests.get(url, timeout=10)
    imagen = Image.open(BytesIO(resp_http.content))

    modelo_vision = genai.GenerativeModel(MODELO_FLASH)
    resp = modelo_vision.generate_content([pregunta, imagen])
    return resp.text

def analizar_imagen_local(ruta: str, pregunta: str) -> str:
    imagen = Image.open(ruta)
    resp = modelo.generate_content([pregunta, imagen])
    return resp.text

# Ejemplo con imagen pública
try:
    url_imagen = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
    descripcion = analizar_imagen_url(url_imagen, '¿Qué ves en esta imagen? Describe brevemente.')
    print('Análisis de imagen:')
    print(descripcion)
except Exception as e:
    print(f'Error (puede ser por falta de API key o conexión): {e}')
    print('Ejemplo de uso:')
    print('  descripcion = analizar_imagen_local(\'ruta/imagen.jpg\', \'¿Qué muestra esta imagen?\')')

## 4. Ventana de contexto larga (1M tokens)

In [ ]:
# Gemini 1.5 Pro tiene ventana de 1M tokens — ideal para documentos muy largos

def analizar_documento_largo(texto: str, pregunta: str) -> str:
    modelo_pro = genai.GenerativeModel(
        MODELO_PRO,
        system_instruction='Eres un experto analizando documentos largos. Sé preciso y cita el número de página cuando sea posible.',
    )
    prompt = f"""Documento completo:
---
{texto}
---

Pregunta: {pregunta}"""

    resp = modelo_pro.generate_content(prompt)
    return resp.text

# Simular documento largo (en producción sería el texto real)
documento_ejemplo = "\n\n".join([
    f"""Capítulo {i}: Introducción a los conceptos de IA generativa — parte {i}.
Los modelos de lenguaje han evolucionado significativamente desde los primeros transformers.
En esta sección exploramos los conceptos de atención multi-cabeza y embeddings posicionales."""
    for i in range(1, 11)
])

try:
    respuesta = analizar_documento_largo(
        documento_ejemplo,
        '¿En qué capítulos se menciona la atención multi-cabeza?'
    )
    print(respuesta)
except Exception as e:
    print(f'Error: {e}')
    print(f'Longitud del documento de prueba: {len(documento_ejemplo.split())} palabras')

## 5. Function calling con Gemini

In [ ]:
import json

# Definir herramientas al estilo Gemini
herramientas_gemini = [
    genai.protos.Tool(
        function_declarations=[
            genai.protos.FunctionDeclaration(
                name='buscar_vuelos',
                description='Busca vuelos disponibles entre dos ciudades.',
                parameters=genai.protos.Schema(
                    type=genai.protos.Type.OBJECT,
                    properties={
                        'origen': genai.protos.Schema(type=genai.protos.Type.STRING),
                        'destino': genai.protos.Schema(type=genai.protos.Type.STRING),
                        'fecha': genai.protos.Schema(type=genai.protos.Type.STRING),
                    },
                    required=['origen', 'destino', 'fecha'],
                ),
            )
        ]
    )
]

def ejecutar_buscar_vuelos(origen: str, destino: str, fecha: str) -> dict:
    # Simulación de búsqueda
    return {
        'vuelos': [
            {'vuelo': 'IB3456', 'precio': 189, 'duracion': '2h15m', 'salida': '08:00'},
            {'vuelo': 'VY1234', 'precio': 134, 'duracion': '2h30m', 'salida': '15:30'},
        ],
        'origen': origen, 'destino': destino, 'fecha': fecha,
    }

modelo_fn = genai.GenerativeModel(MODELO_FLASH, tools=herramientas_gemini)
chat_fn = modelo_fn.start_chat()

try:
    resp = chat_fn.send_message('Busca vuelos de Madrid a Barcelona para el 15 de julio de 2026')

    # Procesar function call si la hay
    if resp.candidates[0].content.parts[0].function_call.name:
        fn_call = resp.candidates[0].content.parts[0].function_call
        args = dict(fn_call.args)
        print(f'Gemini solicita: {fn_call.name}({args})')

        resultado = ejecutar_buscar_vuelos(**args)
        resp_final = chat_fn.send_message(
            genai.protos.Content(
                role='function',
                parts=[genai.protos.Part(
                    function_response=genai.protos.FunctionResponse(
                        name=fn_call.name,
                        response={'result': resultado},
                    )
                )]
            )
        )
        print('\nRespuesta final:')
        print(resp_final.text)
    else:
        print(resp.text)
except Exception as e:
    print(f'Error (puede ser por falta de GOOGLE_API_KEY): {e}')

## 6. Streaming con Gemini

In [ ]:
try:
    stream = modelo.generate_content(
        'Explica paso a paso cómo funciona la atención multi-cabeza en los transformers.',
        stream=True,
    )
    for chunk in stream:
        print(chunk.text, end='', flush=True)
    print()
except Exception as e:
    print(f'Error: {e}')
    print('Para usar Gemini necesitas GOOGLE_API_KEY: https://aistudio.google.com/app/apikey')